<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 1 — Is This Signal, or Noise? (Lancaster Quay Tides)</div>
</div>

#### The opening hook for Lecture 1a, and the dataset for the pandas/matplotlib warm-up in the Week 1 lab.

Source: Rowland Adams, J., Newman, J. & Stefanovska, A. (2023), *"Distinguishing between deterministic oscillations and noise,"* Eur. Phys. J. Spec. Top. 232, 3435–3457. The tidal case study in that paper (their Fig. 4) uses water-level readings from **Lancaster Quay**, on the River Lune.

**How to use this notebook:** run Section 1 and *stop*. Look at the plot before reading Section 2. The point of the exercise is to ask the question honestly.

## <span style="color:#B5121B">1. Real data, without context or interpretation</span>

Below is a water-level record from a real gauge: Lancaster Quay on the River Lune (Environment Agency station `724735`, measure `724735-level-stage-i-15_min-m`). Readings are taken every 15 minutes, in metres.

**Before running Section 2: which parts of the record below are signal, and which are noise?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week1/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

def load_cached_lancaster_quay(path=None):
    # The default. A real extract from the Environment Agency feed below,
    # fetched (24 Sep - 2 Oct 2026) to ensure reproducibility.
    df = pd.read_csv(path or data_path("lancaster_quay_tidal.csv"))
    df["time"] = pd.to_datetime(df["time"])
    return df.sort_values("time").reset_index(drop=True)

def fetch_lancaster_quay(hours=192, timeout=5):
    # Real data -- Environment Agency flood-monitoring API, Lancaster
    # Quay gauge (station 724735, River Lune). Live/historical data.
    # Use this if you want different dates.
    import requests
    from datetime import datetime, timedelta, timezone

    since = (datetime.now(timezone.utc) - timedelta(hours=hours)).strftime("%Y-%m-%dT%H:%M:%SZ")
    url = (
        "https://environment.data.gov.uk/flood-monitoring/id/measures/"
        "724735-level-stage-i-15_min-m/readings"
    )
    resp = requests.get(url, params={"since": since, "_limit": 10000}, timeout=timeout)
    resp.raise_for_status()
    items = resp.json()["items"]
    df = pd.DataFrame(items)[["dateTime", "value"]]
    df["dateTime"] = pd.to_datetime(df["dateTime"])
    df = df.rename(columns={"dateTime": "time", "value": "level_m"}).sort_values("time")
    df = df.reset_index(drop=True)
    return df

def synthetic_lancaster_quay(hours=192, seed=7):
    # SYNTHETIC PLACEHOLDER -- last-resort fallback, only used if neither the
    # cached file nor the live API above is reachable. Mimics the real
    # record's known properties rather than inventing arbitrary numbers:
    #   - M2 tidal period (~12h 25m), the dominant real-world tidal constituent
    #   - a non-sinusoidal shape (harmonics), as the paper's own model for
    #     this gauge assumes: X(t) = A(t) h(theta(t))
    #   - a sensor floor at 2.15 m (below this, the real gauge simply can't
    #     read a level) -- the paper notes this explicitly for this station
    #   - slow (multi-day) amplitude modulation, i.e. spring/neap-style
    #     variation
    rng = np.random.default_rng(seed)
    n = hours * 4  # 15-minute steps
    t_hours = np.arange(n) * 0.25
    period = 12.42  # M2 constituent, hours
    theta = 2 * np.pi * t_hours / period

    # slow spring/neap-style amplitude modulation (~14-day cycle)
    amplitude = 1.5 + 0.3 * np.sin(2 * np.pi * t_hours / (14 * 24) + 0.7)

    # non-sinusoidal shape: fundamental + a couple of harmonics, matching the
    # paper's description of the real Lancaster Quay record
    shape = np.sin(theta) + 0.25 * np.sin(2 * theta + 0.4) + 0.10 * np.sin(3 * theta + 1.1)

    baseline = 4.2
    noise = rng.normal(0, 0.06, size=n)
    level = baseline + amplitude * shape + noise

    # the real gauge can't read below 2.15 m — clip, don't pretend otherwise
    level = np.clip(level, 2.15, None)

    start = pd.Timestamp.now("UTC").floor("15min") - pd.Timedelta(hours=hours)
    time = start + pd.to_timedelta(t_hours, unit="h")
    return pd.DataFrame({"time": time, "level_m": level})


try:
    tide = load_cached_lancaster_quay()
    source = f"real (cached extract, captured up to {tide['time'].max():%Y-%m-%d %H:%M} UTC)"
except Exception as exc:
    try:
        tide = fetch_lancaster_quay(hours=192)
        source = "real (Environment Agency API, live)"
    except Exception as exc2:
        tide = synthetic_lancaster_quay(hours=192)
        source = f"SYNTHETIC placeholder (no cached file, and live API not reached: {exc2!r})"

print(f"Data source: {source}")
print(f"{len(tide)} readings, {tide['time'].min()} to {tide['time'].max()}")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(tide["time"], tide["level_m"], lw=1)
ax.set_xlabel("Time")
ax.set_ylabel("Water level (m)")
ax.set_title("Lancaster Quay, River Lune")
fig.tight_layout()
plt.show()

## <span style="color:#B5121B">2. Explanation and Discussion</span>

This is tidal. The River Lune is tidal at Lancaster Quay, and the water level goes through two highs and two lows a day. This is the M2 lunar semi-diurnal tide, with a period of ≈ 12 h 25 min.

Looking at the raw trace in Section 1 without that knowledge, it's easy to see why some segments might be interpreted as noise:
1. the shape isn't a clean sine wave (the tide isn't symmetric, so it contains harmonics);
2. the amplitude drifts from day to day (spring/neap variation);
3. a naive frequency-domain summary can smear all of this into something that looks broadband and noise-like.

Rowland Adams, Newman & Stefanovska (2023) showed that a classical power-spectrum analysis can mistake this genuinely deterministic, physically-understood oscillation for noise, whereas a time–frequency approach (wavelet transform) correctly identifies it.

Their preliminary model for this gauge is

$$X(t) = A(t)\, h(\theta(t)), \qquad \dot\theta(t) = \frac{2\pi}{12\,\text{h}},$$

a slowly-varying amplitude $A(t)$ and a fixed-frequency phase $\theta(t)$, with $h$ a non-sinusoidal periodic function (hence the harmonics).

You don't need to do anything with this equation. It is the model used to build the synthetic fallback in the loading code above.

## <span style="color:#B5121B">3. Describing the data</span>


In [ ]:
level = tide["level_m"]

print(f"n readings         : {len(level)}")
print(f"mean level         : {level.mean():.3f} m")
print(f"median level       : {level.median():.3f} m")
print(f"std deviation      : {level.std():.3f} m")
print(f"5th / 95th pctile  : {level.quantile(0.05):.3f} m / {level.quantile(0.95):.3f} m")
print(f"minimum reading    : {level.min():.3f} m")

near_min = (level <= level.min() + 0.02).sum()
print(f"\nreadings within 2 cm of that minimum: {near_min} "
      f"({100 * near_min / len(level):.1f}% of the record)")
print("Note that there is a large difference between the mean and median indicating that this isn't a simple symmetric distribution.")

The numbers above summarise the data with single values. They don't tell you about the *shape* of the data.

The histogram below shows the **empirical distribution**: the values this instrument actually recorded. Compare it with the **theoretical distribution** (e.g. a Gaussian) that you might otherwise assume without checking.

In [ ]:
from scipy.stats import skew, kurtosis

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(level, bins=40, color="#B5121B", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Water level (m)")
ax.set_ylabel("Count")
fig.tight_layout()
plt.show()

print(f"skewness        : {skew(level):.2f}  "
      "(0 = symmetric like a Gaussian; positive = long tail toward higher values)")
print(f"excess kurtosis : {kurtosis(level):.2f}  "
      "(0 = Gaussian-like tails; negative = flatter-topped, less peaked)")

This is nothing like a Gaussian. Two features stand out:

1. **A pile-up at the low end.** About half of the readings (378 of 766) are within 2 cm of 2.22–2.23 m. This isn't measurement noise: the water sits at, or very near, a fixed low level for roughly half of each tidal cycle.
2. **A long tail to higher levels.** The remaining readings spread up to the high-tide peaks (~5.5 m). The distribution has a clear **positive skew** (≈ +1.3); a Gaussian has zero skew.

Why isn't it symmetric? This is a *river* gauge, upstream of the open sea. In an estuary the low-water level can be held up by something other than the tide (the river's own flow, a weir downstream, the shape of the channel), while the rising tide can still push the level well above it. That would give a plateau at the bottom and a long tail at the top. This is an inference from the data, not a known fact, so treat it as a question for discussion.

The record is still tidal: the peaks are on average ~12.3 hours apart, close to the M2 period of 12 h 25 min.

The lesson: the *shape* of a distribution is information in its own right. "Roughly Gaussian" is an assumption to check with a histogram.

## <span style="color:#FD9029">4. Optional, ungraded: eyeball the period</span>

Zoom in on the first 48 hours and count the peaks.

In [ ]:
one_day = tide[tide["time"] < tide["time"].min() + pd.Timedelta(hours=48)]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(one_day["time"], one_day["level_m"], marker="o", ms=3, lw=1)
ax.set_xlabel("Time")
ax.set_ylabel("Water level (m)")
ax.set_title("Zoomed in: first 48 hours")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

# A rough, by-eye-style peak count — nothing more sophisticated than that.
from scipy.signal import find_peaks
peak_idx, _ = find_peaks(one_day["level_m"].to_numpy(), distance=4 * 8)  # >=8h apart
peak_times = one_day["time"].to_numpy()[peak_idx]
if len(peak_times) >= 2:
    gaps_hours = np.diff(peak_times).astype("timedelta64[m]").astype(float) / 60
    print(f"{len(peak_times)} peaks found in 48 h; gaps between them (hours): "
          f"{np.round(gaps_hours, 2)}")
    print(f"rough estimated period: {gaps_hours.mean():.2f} h "
          f"(real M2 period is 12.42 h)")
else:
    print(f"{len(peak_times)} peak(s) found — widen the window to see a full cycle.")

## Closing the loop

The tide here is a real, deterministic signal that a careless analysis can mistake for noise. In `week1_foundations.ipynb` we will see how a careless *calculation* can destroy a real, small quantity. 

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>